In [1]:
from ollama import chat
from ollama import ChatResponse
from pydantic import BaseModel, Field
import pandas as pd
from tqdm import tqdm

In [2]:
class Divulgadores(BaseModel):
    name: str = Field(description="Nome do divulgador, exatamente como está no texto.")
    political_party: str = Field(description="Partido político do divulgador")
    is_federal_representative: bool = Field(
        description="Se o divulgador é um representante federal ou não, como Deputado Federal"
    )


class ListaDivulgadores(BaseModel):
    divulgadores: list[Divulgadores] = Field(
        description="As pessoas que divulgaram a notícia."
    )

In [3]:
df = pd.read_csv(
    "./Export-2025-April-30.csv",
    sep=";",
    # encoding="latin1"
)

# # Filtrar por notícias falsas
# df = df[
#     [
#         True if "Falso" in str(t) or "Falsa" in str(t) else False
#         for t in df["tags_0_tag"]
#     ]
# ]

# Pegar lista dos nomes dos deputados

In [4]:
df_parls_raw = pd.read_csv("../../../../data/df_parlamentares_por_legislatura.csv")

df_parls = df_parls_raw[(df_parls_raw['idLegislatura'] == 56) & (df_parls_raw['nome'].notna())]

df_parls = df_parls[['id', 'nome']].drop_duplicates()

In [5]:
def get_divulgadores(content: str, it: int = 0, max_it: int = 3) -> list[Divulgadores]:
    response: ChatResponse = chat(
        model="deepseek-r1:7b",
        format=ListaDivulgadores.model_json_schema(),
        options={
            "temperature": 0.0,
        },
        messages=[
            {
                "role": "user",
                "content": f"""Dado o conteúdo da notícia, identifique as **pessoas** que divulgaram o conteúdo falso.

                **IMPORTANTE**:
                - Diferencie entre quem divulgou a notíticia de quem foi o alvo da notícia
                - Um divulgador é a pessoa que compartilhou a notícia
                - Retorne apenas quem divulgou, se não houver divulgador, retorne uma lista vazia
                - Considere apenas os nomes que estão na lista abaixo:

                **LISTA DE DIVULGADORES POSSÍVEIS**:
                {df_parls.to_dict(orient="records")}

                **CONTEÚDO**: {content}

                """,
            },
        ],
    )

    try:
        divulgadores = ListaDivulgadores.model_validate_json(response.message.content)
        return divulgadores
    except Exception as e:
        if it < max_it:
            return get_divulgadores(content, it + 1, max_it)
        else:
            return []


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading


df_name = "./divulgadores_3.csv"
df_divulgadores = pd.read_csv(df_name)
data = df_divulgadores.to_dict("records")


lock = threading.Lock()


def process_content(content, index_: int, it: int = 0, max_it: int = 3):
    seen_indexes = [d["index_"] for d in data]
    if index_ in seen_indexes:
        return None
    res = get_divulgadores(content, it, max_it)
    return {"content": content, "divulgadores": res, "index_": index_}


# Use ThreadPoolExecutor for parallel processing
with ThreadPoolExecutor(max_workers=4) as executor:
    # Submit all tasks
    future_to_content = {
        executor.submit(process_content, content["Content"], index_): content
        for index_, content in df[df["Content"].notna()].iterrows()
    }

    # Process completed tasks with progress bar
    for future in tqdm(as_completed(future_to_content), total=len(future_to_content)):
        result = future.result()
        with lock:
            if result is not None:
                data.append(result)
                df = pd.DataFrame(data)
                df.to_csv(df_name, index=False)


 56%|█████▌    | 818/1459 [8:07:55<262:55:48, 1476.67s/it]